In [ ]:
%pip install pylatexenc
%pip install matplotlib
#%pip install git+https://opencode.it4i.eu/adas-private/hpc-q-integration/qaas.git@main

In [ ]:
import os
# os.environ["QPROVIDER_LOGLEVEL"] = "DEBUG"
os.environ["QPROVIDER_LOGLEVEL"] = "INFO"

# Connect to system

In [ ]:
# Setup Lexis session
from py4lexis.session import LexisSession

lexis_session = LexisSession()

In [ ]:
token = lexis_session.get_access_token()
lexis_project = "vlq_demo_project"
resource_name = "VLQ-CZ" 

In [ ]:
from qaas import QProvider, QBackend, QJob

provider = QProvider(token, lexis_project)
# QBackend
backend:QBackend = provider.get_backend(resource_name)

print(f'Qubit: {backend.architecture.qubits}')

print(f'Gates: {backend.architecture.gates.keys()}')

# Define Quantum Circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(15,5)
for i in range(15): qc.h(i)
for i in range(14): qc.cx(i,i+1)
qc.cx(0,14)
qc.measure(range(5), range(5))


# Transpile Circuit

In [ ]:
from qaas.backend_iqm import transpile_to_IQM

# Transpile circuit Locally
qc_transpiled_local = transpile_to_IQM(qc, backend, optimize_single_qubits=False)

# Run Circuit

In [ ]:
from time import time
# Run circuit with number of SHOTS
SHOTS = 100

client_execution_time = time()

# Passing as argument not transpiled circuit
job:QJob = backend.run([qc_transpiled_local], shots=SHOTS)
result = job.result()
client_execution_time = time()-client_execution_time
results_dict = result.get_counts()
print("Raw counts:", results_dict)

# Print non-zero results
for key, count in results_dict.items():
    if count > 0:
        print(f"State '{key}': {count} counts")
        # Only convert to int if key is actually a binary string
        if all(c in '01' for c in key):
            print(f"  -> Decimal value: {int(key, 2)}")


In [ ]:
print(f"Initialization time on remote {job.remote_initialization_runtime} s")
print(f"Circuit runtime {job.remote_iqm_client_job_runtime} s")
print(f"Real HW execution {job.remote_hw_runtime} s")
print(f"Overall runtime on remote {job.remote_backend_runtime} s")
print("-"*40)

print(f"QProvider runtime ended in {job.qaas_runtime} s")
print(f"QProvider fetched outputs in {job.qaas_fetching_runtime} s")
print(f"Pure client execution time {client_execution_time} s")

In [ ]:
for entry in result.timeline:
    if entry.status in ["execution_started", "execution_ended"]:
        print(entry)


In [ ]:
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# Plot histogram
plot_histogram(results_dict, figsize=(12, 6), bar_labels=False)  # hide text above bars